# Paper Leak Analysis
## Data Extraction, Validation and Cleaning

This notebook covers the Data Extraction, Data Validation and Data Cleaning stages of the project.


## 1. Import Libraries


In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)


## 2. Load the Raw Dataset


In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


# 3. Data Extraction

The extraction tables below organize the information needed for the later analysis stage. They are not separate cleaned datasets; they are views of the raw data prepared for different types of questions.


## 3.1 Location and Exam-based Extraction

This table contains the main fields needed for location, exam, era and conducting-body based questions.


In [ ]:
location_exam_data = df[
    [
        "incident_id",
        "area",
        "exam_name",
        "era",
        "conducting_body",
        "body_type",
        "leak_status"
    ]
].copy()

display(location_exam_data.head(10))


### Location-wise Incident Summary


In [ ]:
location_summary = (
    df.groupby("area", dropna=False)
      .agg(
          incident_count=("incident_id", "nunique"),
          exams=("exam_name", lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))),
          conducting_bodies=("conducting_body", lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))),
          eras=("era", lambda x: ", ".join(sorted(x.dropna().astype(str).unique())))
      )
      .reset_index()
)

display(location_summary)


### States/Locations with Incidents in Both UPA and NDA Eras


In [ ]:
era_by_location = (
    df.dropna(subset=["area", "era"])
      .groupby("area")["era"]
      .nunique()
      .reset_index(name="era_count")
)

both_eras = era_by_location[era_by_location["era_count"] > 1]

display(both_eras)


### Locations with Multiple Conducting Bodies


In [ ]:
multiple_bodies = (
    df.dropna(subset=["area", "conducting_body"])
      .groupby("area")["conducting_body"]
      .nunique()
      .reset_index(name="conducting_body_count")
)

multiple_bodies = multiple_bodies[multiple_bodies["conducting_body_count"] > 1]

display(multiple_bodies)


### All India Level Incidents


In [ ]:
all_india_incidents = df[
    df["area"].astype(str).str.strip().str.lower().eq("all india")
][
    ["incident_id", "date", "exam_name", "conducting_body", "leak_status"]
]

display(all_india_incidents)


## 3.2 Confidence-based Extraction

This table can be used to answer questions about High, Medium and Low confidence cases.


In [ ]:
confidence_data = (
    df[
        ["incident_id", "area", "exam_name", "confidence", "leak_status"]
    ]
    .copy()
)

confidence_summary = (
    confidence_data.groupby(["area", "confidence"], dropna=False)
    .size()
    .reset_index(name="incident_count")
    .sort_values(["area", "confidence"])
)

display(confidence_summary)


## 3.3 Conducting Body-based Extraction

This table uses both `conducting_body` and `body_type` to support Central and State conducting-body questions.


In [ ]:
conducting_body_data = df[
    [
        "incident_id",
        "area",
        "exam_name",
        "conducting_body",
        "body_type"
    ]
].copy()

conducting_body_summary = (
    conducting_body_data.groupby(
        ["area", "body_type", "conducting_body"],
        dropna=False
    )
    .size()
    .reset_index(name="incident_count")
)

display(conducting_body_summary)


### Central Conducting Body Incidents


In [ ]:
central_incidents = conducting_body_data[
    conducting_body_data["body_type"].eq("Central")
]

display(central_incidents)


### State Conducting Body Incidents


In [ ]:
state_incidents = conducting_body_data[
    conducting_body_data["body_type"].eq("State")
]

display(state_incidents)


## 3.4 Legal Action / Action Taken Extraction

This table keeps arrests, convictions and action taken together so that the later analysis can answer different legal-action questions from one table.


In [ ]:
legal_action_data = df[
    [
        "incident_id",
        "area",
        "exam_name",
        "action_taken",
        "arrests",
        "convictions"
    ]
].copy()

display(legal_action_data.head(10))


### Incidents with Arrests


In [ ]:
arrest_cases = legal_action_data[
    legal_action_data["arrests"].notna() &
    (legal_action_data["arrests"] > 0)
]

display(arrest_cases)


### Incidents with Convictions


In [ ]:
conviction_cases = legal_action_data[
    legal_action_data["convictions"].notna() &
    (legal_action_data["convictions"] > 0)
]

display(conviction_cases)


### Incidents with Reported Legal Action


In [ ]:
legal_action_cases = legal_action_data[
    (legal_action_data["arrests"].fillna(0) > 0) |
    (legal_action_data["convictions"].fillna(0) > 0)
]

display(legal_action_cases)


## 3.5 Era-based Extraction

This table provides a compact state/location and era view for later comparison.


In [ ]:
era_data = df[
    ["incident_id", "area", "era", "exam_name"]
].copy()

era_summary = (
    era_data.groupby(["area", "era"], dropna=False)
    .size()
    .reset_index(name="incident_count")
)

display(era_summary)


# 4. Data Validation

The validation is performed on the original dataset before cleaning.


## 4.1 Missing Values and Missing Percentage


In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
})

display(
    missing_report[missing_report["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)


## 4.2 Duplicate Records and Incident IDs


In [ ]:
print("Duplicate records:", df.duplicated().sum())
print("Duplicate incident IDs:", df["incident_id"].duplicated().sum())


## 4.3 Validate Incident ID Format


In [ ]:
valid_id = df["incident_id"].astype(str).str.fullmatch(r"PL-\d{4}")

print("Invalid incident IDs:", (~valid_id).sum())

if (~valid_id).any():
    display(df.loc[~valid_id, ["incident_id"]])


## 4.4 Validate Date Format and Range


In [ ]:
parsed_dates = pd.to_datetime(df["date"], errors="coerce")

print("Invalid date values:", parsed_dates.isna().sum())

outside_range = (
    parsed_dates.notna() &
    ((parsed_dates.dt.year < 2004) | (parsed_dates.dt.year > 2026))
)

print("Dates outside expected range:", outside_range.sum())

if outside_range.any():
    display(df.loc[outside_range, ["incident_id", "date"]])


## 4.5 Validate Categorical Values


In [ ]:
expected_values = {
    "era": {"UPA (2004-May2014)", "NDA (May2014-now)"},
    "body_type": {"Central", "State"},
    "leak_status": {"Confirmed", "Alleged", "Denied", "Suspected"},
    "confidence": {"High", "Medium", "Low"}
}

for column, allowed_values in expected_values.items():
    invalid = ~df[column].isin(allowed_values) & df[column].notna()
    print(f"{column} - invalid values:", invalid.sum())

    if invalid.any():
        display(df.loc[invalid, ["incident_id", column]])


## 4.6 Check Blank Text Values and Extra Whitespace


In [ ]:
text_columns = df.select_dtypes(include="object").columns

blank_report = {
    column: df[column].fillna("").astype(str).str.strip().eq("").sum()
    for column in text_columns
}

whitespace_report = {
    column: (
        df[column].dropna().astype(str).str.len() !=
        df[column].dropna().astype(str).str.strip().str.len()
    ).sum()
    for column in text_columns
}

print("Blank text values:")
display(pd.Series(blank_report, name="blank_values")[lambda x: x > 0])

print("Values with extra whitespace:")
display(pd.Series(whitespace_report, name="extra_whitespace")[lambda x: x > 0])


## 4.7 Validate Numerical Data


In [ ]:
numeric_columns = [
    "arrests",
    "convictions",
    "aspirants_affected",
    "linked_deaths"
]

print("Data types:")
display(df[numeric_columns].dtypes)

negative_values = {
    column: (df[column] < 0).sum()
    for column in numeric_columns
}

print("Negative values:")
display(pd.Series(negative_values, name="negative_count"))


## 4.8 Check Numerical Consistency


In [ ]:
conviction_issue = (
    df["arrests"].notna() &
    df["convictions"].notna() &
    (df["convictions"] > df["arrests"])
)

print("Records where convictions are greater than arrests:", conviction_issue.sum())

if conviction_issue.any():
    display(df.loc[
        conviction_issue,
        ["incident_id", "arrests", "convictions"]
    ])


## 4.9 Validate Linked Deaths Information


In [ ]:
death_note_issue = (
    df["linked_deaths"].notna() &
    (df["linked_deaths"] > 0) &
    df["deaths_note"].isna()
)

print("Positive linked deaths without a note:", death_note_issue.sum())


## 4.10 Validate Source Information


In [ ]:
blank_source = (
    df["source_name"].isna() |
    df["source_name"].astype(str).str.strip().eq("")
)

valid_url = df["source_url"].astype(str).str.fullmatch(
    r"https?://\S+",
    na=False
)

print("Blank source names:", blank_source.sum())
print("Invalid source URLs:", (~valid_url).sum())
print("Repeated source URLs:", df["source_url"].duplicated().sum())


# 5. Data Cleaning

The original `df` is kept unchanged. Cleaning is performed on a separate copy.


In [ ]:
cleaned_df = df.copy()


## 5.1 Remove Duplicate Records


In [ ]:
before = len(cleaned_df)

cleaned_df = cleaned_df.drop_duplicates().copy()

print("Duplicate records removed:", before - len(cleaned_df))


## 5.2 Standardize Text and Categorical Values


In [ ]:
text_columns = cleaned_df.select_dtypes(include="object").columns

for column in text_columns:
    cleaned_df[column] = cleaned_df[column].str.strip()

categorical_columns = [
    "era",
    "body_type",
    "leak_status",
    "confidence"
]

for column in categorical_columns:
    cleaned_df[column] = cleaned_df[column].str.strip()

print("Text and categorical formatting standardized.")


## 5.3 Standardize Action Taken


In [ ]:
cleaned_df["action_taken"] = cleaned_df["action_taken"].str.replace(
    r"\s*\+\s*",
    " + ",
    regex=True
)

print("Action taken formatting standardized.")


## 5.4 Convert Date and Numerical Columns


In [ ]:
cleaned_df["date"] = pd.to_datetime(
    cleaned_df["date"],
    errors="coerce"
)

for column in numeric_columns:
    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column],
        errors="coerce"
    )

print("Date and numerical columns converted.")


## 5.5 Handle Missing Values

Missing numerical values are retained as missing because the source may not have reported the information. They are not changed to zero without evidence.


In [ ]:
print("Missing values retained for fields where information was not reported:")
display(
    cleaned_df[
        ["arrests", "convictions", "aspirants_affected", "linked_deaths"]
    ].isna().sum()
)


## 5.6 Handle Invalid Values

Values that need source-level verification are not changed automatically. This avoids introducing unsupported assumptions into the dataset.


In [ ]:
review_cases = (
    cleaned_df["arrests"].notna() &
    cleaned_df["convictions"].notna() &
    (cleaned_df["convictions"] > cleaned_df["arrests"])
)

print("Records requiring source review:", review_cases.sum())

if review_cases.any():
    display(cleaned_df.loc[
        review_cases,
        ["incident_id", "arrests", "convictions"]
    ])


# 6. Final Validation


## 6.1 Missing Values After Cleaning


In [ ]:
final_missing = pd.DataFrame({
    "missing_count": cleaned_df.isnull().sum(),
    "missing_percentage": (cleaned_df.isnull().mean() * 100).round(2)
})

display(
    final_missing[final_missing["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)


## 6.2 Duplicate Records After Cleaning


In [ ]:
print("Duplicate records:", cleaned_df.duplicated().sum())
print("Duplicate incident IDs:", cleaned_df["incident_id"].duplicated().sum())


## 6.3 Data Types After Cleaning


In [ ]:
display(cleaned_df.dtypes)


## 6.4 Final Dataset Dimensions


In [ ]:
print("Final rows:", cleaned_df.shape[0])
print("Final columns:", cleaned_df.shape[1])


## 6.5 Display the Cleaned Dataset


In [ ]:
display(cleaned_df.head(10))


## 6.6 Save the Cleaned Dataset


In [ ]:
cleaned_df.to_csv("paper_leaks_cleaned.csv", index=False)

print("Cleaned dataset saved as paper_leaks_cleaned.csv")


# End of Data Extraction, Validation and Cleaning

The cleaned dataset can now be passed to the next stages of the project for Data Aggregation, Representation, Analysis, Visualization and Results Interpretation.
